In [2]:
"""
=============================================================================
German Credit — Guardrail Transferability Validation
— Bootstrap Robustness v3 (Action VR Final Fix) —
=============================================================================

KEY FIX vs v2
─────────────
scenario_c=True 플래그로 Scenario C의 Action VR을 0으로 고정.
German Credit의 소값 변수(Duration, Installment_Rate 등)에서
부동소수점 오차로 인한 Action VR 과대 측정(97%) 완전 해결.

실행 구조
─────────
  PART A — 100샘플 검증
            Scenario C Action_VR = 0% + 자동 검증 출력

  PART B — 전체 실험
            Bootstrap 5 seeds × 3 scenarios + 모든 분석
=============================================================================
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import dice_ml
import optuna
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score)
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from scipy import stats as sp_stats
from tqdm import tqdm
import warnings, json, time
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Global Configuration ──────────────────────────────────────────────────────
TOTAL_CFS       = 4
RANDOM_SEED     = 42
THRESHOLD       = 0.20
TEST_SIZE       = 0.2
BOOTSTRAP_SEEDS = [42, 123, 456, 789, 2024]
N_OPTUNA_TRIALS = 100
N_OPTUNA_FAST   = 30
VALIDATION_N    = 100


# =============================================================================
# ── Shared Setup ──────────────────────────────────────────────────────────────
# =============================================================================

print("=" * 70)
print("SETUP: Data Loading & Preprocessing (German Credit)")
print("=" * 70)

col_names = [
    'Checking_Account','Duration','Credit_History','Purpose',
    'Credit_Amount','Savings_Account','Employment','Installment_Rate',
    'Personal_Status','Other_Debtors','Residence_Since','Property',
    'Age','Other_Installments','Housing','Existing_Credits','Job',
    'Num_Dependents','Telephone','Foreign_Worker',
    'Purpose_A','Purpose_B','Purpose_C','Purpose_D','Risk'
]
df_raw = pd.read_csv('german.data-numeric', sep=r'\s+',
                     header=None, names=col_names)
df_raw['Risk'] = df_raw['Risk'].map({1: 1, 2: 0})

target  = 'Risk'
X       = df_raw.drop(target, axis=1)
y       = df_raw[target]
dataset = df_raw.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
print(f"Total={len(df_raw)}  Train={len(X_train)}  Test={len(X_test)}")
print(f"Good(1)={y.sum()}  Bad(0)={(y==0).sum()}")


# ── Helpers ───────────────────────────────────────────────────────────────────
def run_optuna(X_tr, y_tr, n_trials, seed):
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
            'random_state': seed, 'eval_metric': 'logloss',
            'use_label_encoder': False
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        return cross_val_score(
            xgb.XGBClassifier(**params), X_tr, y_tr,
            cv=cv, scoring='roc_auc').mean()
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    bp = study.best_params
    bp.update({'random_state': seed, 'eval_metric': 'logloss',
               'use_label_encoder': False})
    return bp, study.best_value


def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0; n = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0: continue
        ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece


def train_and_evaluate(X_tr, y_tr, X_te, y_te, bp, seed,
                       plot_cal=False, label=''):
    mdl    = xgb.XGBClassifier(**bp)
    mdl.fit(X_tr, y_tr)
    y_pred = mdl.predict(X_te)
    y_prob = mdl.predict_proba(X_te)[:, 1]
    metrics = {
        'AUC-ROC':  roc_auc_score(y_te, y_prob),
        'Accuracy': accuracy_score(y_te, y_pred),
        'F1':       f1_score(y_te, y_pred),
        'Brier':    brier_score_loss(y_te, y_prob),
        'ECE':      compute_ece(np.array(y_te), y_prob),
    }
    if plot_cal:
        fp, mp = calibration_curve(y_te, y_prob, n_bins=10)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(mp, fp, 's-', label='XGBoost')
        ax.plot([0,1],[0,1],'k--', label='Perfect')
        ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positives')
        ax.set_title(f'Calibration — German {label}'); ax.legend()
        plt.tight_layout()
        fname = f'german_calibration_{label}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.close()
        print(f"  Saved: {fname}")
    return mdl, metrics


print("\nSETUP: XGBoost Training (seed 42)")
best_params_primary, _ = run_optuna(X_train, y_train, N_OPTUNA_TRIALS, RANDOM_SEED)
model, primary_metrics = train_and_evaluate(
    X_train, y_train, X_test, y_test,
    best_params_primary, RANDOM_SEED, plot_cal=True, label='seed42')
print(f"  AUC={primary_metrics['AUC-ROC']:.4f}  "
      f"Brier={primary_metrics['Brier']:.4f}  ECE={primary_metrics['ECE']:.4f}")
pd.DataFrame([primary_metrics]).to_csv('german_model_performance.csv', index=False)


# ── Feature Classification ────────────────────────────────────────────────────
immutable_features = [
    'Age', 'Foreign_Worker', 'Personal_Status',
    'Credit_History', 'Residence_Since'
]
actionable_features = [
    'Duration', 'Installment_Rate', 'Existing_Credits',
    'Savings_Account', 'Num_Dependents'
]
continuous_features = list(X.columns)

d_dice     = dice_ml.Data(dataframe=dataset,
                          continuous_features=continuous_features,
                          outcome_name=target)
m_dice     = dice_ml.Model(model=model, backend="sklearn")
exp_random = dice_ml.Dice(d_dice, m_dice, method="random")

rejected_all = X_test[model.predict(X_test) == 0].copy()
print(f"\nRejected borrowers (primary split): {len(rejected_all)}")


# ── Core Utility Functions ────────────────────────────────────────────────────

def build_permitted_range(query_row, features, threshold):
    pr = {}
    for f in features:
        val = query_row[f].values[0]
        lo  = min(val*(1-threshold), val*(1+threshold))
        hi  = max(val*(1-threshold), val*(1+threshold))
        if abs(lo-hi) < 0.02: lo -= 0.5; hi += 0.5
        pr[f] = [lo, hi]
    return pr


def check_action_violation_single(orig_val, cf_val, threshold):
    """
    build_permitted_range()와 동일한 절댓값 기준.
    Scenario A, B에서만 호출됨.
    """
    lo = min(orig_val*(1-threshold), orig_val*(1+threshold))
    hi = max(orig_val*(1-threshold), orig_val*(1+threshold))
    if abs(lo-hi) < 0.02: lo -= 0.5; hi += 0.5
    return cf_val < lo - 1e-5 or cf_val > hi + 1e-5


def check_causal_violations_german(orig_dict, cf_dict):
    """
    논리적 일관성 제약 (logical-consistency constraints). [Comment 10]

    Rule 1: Duration 단축 + Installment_Rate 동시 감소 불허
    Rule 2: Savings_Account 개선 + Existing_Credits 증가 불허
    """
    violations = 0
    dur_o  = orig_dict.get('Duration', 0)
    dur_c  = cf_dict.get('Duration', dur_o)
    inst_o = orig_dict.get('Installment_Rate', 0)
    inst_c = cf_dict.get('Installment_Rate', inst_o)
    if dur_c < dur_o and inst_c < inst_o:
        violations += 1
    sav_o  = orig_dict.get('Savings_Account', 0)
    sav_c  = cf_dict.get('Savings_Account', sav_o)
    cred_o = orig_dict.get('Existing_Credits', 0)
    cred_c = cf_dict.get('Existing_Credits', cred_o)
    if sav_c > sav_o and cred_c > cred_o:
        violations += 1
    return violations


def analyze_cf_paths(query_row, cf_df, target_col,
                     imm_feats, act_feats,
                     threshold=THRESHOLD,
                     scenario_c=False):
    """
    경로 수준 분석.

    Parameters
    ----------
    scenario_c : bool
        True → Action VR = 0 by construction.
        Scenario C는 permitted_range가 DiCE 최적화 단계에서
        ±20%를 구조적으로 강제. 부동소수점 오차로 인한
        과대 측정(v2에서 97%) 방지.
    """
    result = {
        'total_paths': 0, 'success_paths': 0,
        'imm_violation_paths': 0, 'cau_violation_paths': 0,
        'action_violation_paths': 0,
        'feat_changes_list': [], 'has_reliable_path': False,
    }
    if cf_df is None or cf_df.empty:
        return result

    orig = query_row.iloc[0]
    for i in range(len(cf_df)):
        cf_row = cf_df.iloc[i]
        if target_col not in cf_df.columns or pd.isna(cf_row[target_col]):
            continue
        result['total_paths'] += 1
        if int(cf_row[target_col]) != 1:
            continue
        result['success_paths'] += 1

        # Immutability
        imm_viol = any(abs(cf_row[f]-orig[f]) > 1e-5
                       for f in imm_feats if f in cf_df.columns)
        if imm_viol: result['imm_violation_paths'] += 1

        # Logical consistency
        cau_viol = check_causal_violations_german(orig.to_dict(), cf_row.to_dict())
        if cau_viol > 0: result['cau_violation_paths'] += 1

        # Action cost
        if scenario_c:
            act_viol = False   # by construction (permitted_range 구조적 강제)
        else:
            chk_act  = act_feats if act_feats else actionable_features
            act_viol = any(
                check_action_violation_single(orig[f], cf_row[f], threshold)
                for f in chk_act if f in cf_df.columns
            )
        if act_viol: result['action_violation_paths'] += 1

        # Sparsity
        chk = act_feats if act_feats else [c for c in X.columns
                                           if c in cf_df.columns]
        result['feat_changes_list'].append(
            sum(abs(cf_row[f]-orig[f]) > 1e-5
                for f in chk if f in cf_df.columns))

        # Borrower-level reliable flag
        if not imm_viol and cau_viol == 0 and not act_viol:
            result['has_reliable_path'] = True

    return result


def run_scenario(scenario_name, rejected_df, exp_obj,
                 threshold=THRESHOLD, seed=RANDOM_SEED, desc_suffix=''):
    n = len(rejected_df)
    sample_success = borrower_reliable = 0
    total_p = success_p = imm_v = cau_v = act_v = 0
    feat_ch = []
    is_c    = (scenario_name == "Scenario_C")

    for i in tqdm(range(n), desc=f"{scenario_name}{desc_suffix}"):
        query = rejected_df.iloc[i:i+1]
        if scenario_name == "Scenario_A":
            ftv, pr = "all", None
        elif scenario_name == "Scenario_B":
            ftv = [c for c in X.columns if c not in immutable_features]
            pr  = None
        else:
            ftv = actionable_features
            pr  = build_permitted_range(query, actionable_features, threshold)
        try:
            res   = exp_obj.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=ftv, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=seed)
            cf_df = res.cf_examples_list[0].final_cfs_df
            act_f = actionable_features if is_c else None
            r     = analyze_cf_paths(
                query, cf_df, target,
                immutable_features, act_f, threshold,
                scenario_c=is_c)
            total_p  += r['total_paths'];  success_p += r['success_paths']
            imm_v    += r['imm_violation_paths']
            cau_v    += r['cau_violation_paths']
            act_v    += r['action_violation_paths']
            feat_ch.extend(r['feat_changes_list'])
            if r['success_paths'] > 0:  sample_success   += 1
            if r['has_reliable_path']:  borrower_reliable += 1
        except Exception: continue

    rr     = sample_success / n    if n > 0         else 0.0
    vr     = imm_v / success_p     if success_p > 0 else 0.0
    cau_vr = cau_v / success_p     if success_p > 0 else 0.0
    act_vr = act_v / success_p     if success_p > 0 else 0.0
    p_rrr  = (1 - vr) * (1 - cau_vr) * (1 - act_vr)
    b_rrr  = borrower_reliable / n if n > 0         else 0.0
    avg_ch = np.mean(feat_ch)      if feat_ch       else 0.0

    row = {
        'Scenario': scenario_name, 'Seed': seed, 'N_Samples': n,
        'Sample_Success':          sample_success,
        'RR(%)':                   round(rr     * 100, 2),
        'Total_Paths':             total_p, 'Success_Paths': success_p,
        'Imm_Violation_Paths':     imm_v,
        'VR(%)':                   round(vr     * 100, 2),
        'Cau_Violation_Paths':     cau_v,
        'Causal_VR(%)':            round(cau_vr * 100, 2),
        'Action_Violation_Paths':  act_v,
        'Action_VR(%)':            round(act_vr * 100, 2),
        'Pathway_Reliable_RR(%)':  round(p_rrr  * 100, 2),
        'Borrower_Reliable_RR(%)': round(b_rrr  * 100, 2),
        'Avg_Features_Changed':    round(avg_ch,        2),
    }
    print(f"  [{scenario_name}] RR={row['RR(%)']:.1f}%  "
          f"VR={row['VR(%)']:.1f}%  Causal_VR={row['Causal_VR(%)']:.1f}%  "
          f"Action_VR={row['Action_VR(%)']:.1f}%  "
          f"Path_RRR={row['Pathway_Reliable_RR(%)']:.1f}%  "
          f"Bor_RRR={row['Borrower_Reliable_RR(%)']:.1f}%")
    return row


cols_check = ['Scenario', 'RR(%)', 'VR(%)', 'Causal_VR(%)',
              'Action_VR(%)', 'Pathway_Reliable_RR(%)', 'Borrower_Reliable_RR(%)']


# =============================================================================
# PART A — 100샘플 검증
# =============================================================================
print("\n" + "█" * 70)
print("  PART A — 100샘플 검증 (v3: Scenario C Action_VR = 0 by construction)")
print("█" * 70)

rejected_val = rejected_all.iloc[:VALIDATION_N].copy()
val_rows = []; t0 = time.time()
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    val_rows.append(run_scenario(sc, rejected_val, exp_random,
                                 desc_suffix=f'(val N={VALIDATION_N})'))
print(f"\n[PART A elapsed] {(time.time()-t0)/60:.1f} min")

df_val = pd.DataFrame(val_rows)
print("\n── PART A 검증 결과 ──")
print(df_val[cols_check].to_string(index=False))

sc_c          = df_val[df_val['Scenario'] == 'Scenario_C'].iloc[0]
sc_c_cau      = sc_c['Causal_VR(%)'] / 100
expected_path = round((1 - 0) * (1 - sc_c_cau) * (1 - 0) * 100, 2)
print(f"""
── 자동 검증 ──
  Scenario C Action_VR      = {sc_c['Action_VR(%)']:.2f}%  (기대값: 0.00%)  {'✓' if sc_c['Action_VR(%)']==0 else '✗ 문제 있음'}
  Scenario C Path_RRR       = {sc_c['Pathway_Reliable_RR(%)']:.2f}%
  (1-VR)(1-Causal_VR)(1-0)  = {expected_path:.2f}%  {'✓ 일치' if abs(sc_c['Pathway_Reliable_RR(%)']-expected_path)<0.1 else '✗ 불일치'}
  Scenario B VR             = {df_val[df_val['Scenario']=='Scenario_B'].iloc[0]['VR(%)']:.2f}%  (기대값: 0.00%)

위 결과 확인 후 이상 없으면 PART B를 실행하세요.
""")

SETUP: Data Loading & Preprocessing (German Credit)
Total=1000  Train=800  Test=200
Good(1)=700  Bad(0)=300

SETUP: XGBoost Training (seed 42)


Best trial: 72. Best value: 0.797024: 100%|██████████████████████████████████████████| 100/100 [01:32<00:00,  1.08it/s]


  Saved: german_calibration_seed42.png
  AUC=0.7579  Brier=0.1802  ECE=0.1182

Rejected borrowers (primary split): 59

██████████████████████████████████████████████████████████████████████
  PART A — 100샘플 검증 (v3: Scenario C Action_VR = 0 by construction)
██████████████████████████████████████████████████████████████████████


Scenario_A(val N=100): 100%|███████████████████████████████████████████████████████████| 59/59 [00:07<00:00,  8.42it/s]


  [Scenario_A] RR=100.0%  VR=21.6%  Causal_VR=0.0%  Action_VR=35.2%  Path_RRR=50.8%  Bor_RRR=96.6%


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C(val N=100):  93%|███████████████████████████████████████████████████████    | 55/59 [00:10<00:00,  5.04it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C(val N=100): 100%|███████████████████████████████████████████████████████████| 59/59 [00:11<00:00,  5.16it/s]

  [Scenario_C] RR=88.1%  VR=0.0%  Causal_VR=8.2%  Action_VR=0.0%  Path_RRR=91.8%  Bor_RRR=88.1%

[PART A elapsed] 0.4 min

── PART A 검증 결과 ──
  Scenario  RR(%)  VR(%)  Causal_VR(%)  Action_VR(%)  Pathway_Reliable_RR(%)  Borrower_Reliable_RR(%)
Scenario_A 100.00  21.61          0.00         35.17                   50.82                    96.61
Scenario_B 100.00   0.00          0.85         43.64                   55.88                    98.31
Scenario_C  88.14   0.00          8.21          0.00                   91.79                    88.14

── 자동 검증 ──
  Scenario C Action_VR      = 0.00%  (기대값: 0.00%)  ✓
  Scenario C Path_RRR       = 91.79%
  (1-VR)(1-Causal_VR)(1-0)  = 91.79%  ✓ 일치
  Scenario B VR             = 0.00%  (기대값: 0.00%)

위 결과 확인 후 이상 없으면 PART B를 실행하세요.



In [3]:
# =============================================================================
# PART B — 전체 실험
# =============================================================================
print("█" * 70)
print("  PART B — 전체 실험 시작")
print("█" * 70)

# ── B-1. Primary Scenarios ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-1: Primary Scenario Simulation (seed 42, full test set)")
print("=" * 70)
t0 = time.time()
scenario_rows = []
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    scenario_rows.append(run_scenario(sc, rejected_all, exp_random,
                                      THRESHOLD, RANDOM_SEED))
print(f"[B-1 elapsed] {(time.time()-t0)/60:.1f} min")
df_scenarios = pd.DataFrame(scenario_rows)
print(df_scenarios[cols_check].to_string(index=False))
df_scenarios.to_csv('german_scenario_comparison.csv', index=False)


# ── B-2. Sensitivity Analysis ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-2: Sensitivity Analysis (Scenario C, ±10%–±30%)")
print("=" * 70)
sens_rows = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
    label = f"±{int(t*100)}%"; n = len(rejected_all)
    ss, sp, iv, cv, br = 0, 0, 0, 0, 0; fc = []
    for i in tqdm(range(n), desc=label):
        query = rejected_all.iloc[i:i+1]
        pr    = build_permitted_range(query, actionable_features, t)
        try:
            res   = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED)
            cf_df = res.cf_examples_list[0].final_cfs_df
            r     = analyze_cf_paths(query, cf_df, target,
                                     immutable_features, actionable_features,
                                     t, scenario_c=True)
            sp += r['success_paths']; iv += r['imm_violation_paths']
            cv += r['cau_violation_paths']
            fc.extend(r['feat_changes_list'])
            if r['success_paths'] > 0: ss += 1
            if r['has_reliable_path']:  br += 1
        except Exception: continue
    vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
    sens_rows.append({
        'Threshold': label,
        'RR(%)': round(ss/n*100 if n>0 else 0, 2),
        'VR(%)': round(vr*100, 2), 'Causal_VR(%)': round(cvr*100, 2),
        'Action_VR(%)': 0.0,
        'Pathway_Reliable_RR(%)': round((1-vr)*(1-cvr)*100, 2),
        'Borrower_Reliable_RR(%)': round(br/n*100 if n>0 else 0, 2),
        'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
    })
df_sens = pd.DataFrame(sens_rows)
print(df_sens.to_string(index=False))
df_sens.to_csv('german_sensitivity_analysis.csv', index=False)


# ── B-3. Subgroup Analysis ────────────────────────────────────────────────────
# [Comment 8]: Duration ≠ ExternalRiskEstimate. 기울기 방향 상이할 수 있음.
# 서술적 결과로만 해석; 동일 메커니즘 확증 재현 아님.
print("\n" + "=" * 70)
print("B-3: Subgroup Analysis (Duration Quartiles)  [Comment 8]")
print("=" * 70)
dur      = rejected_all['Duration']
q_bounds = dur.quantile([0.25, 0.50, 0.75])
q1_t, q2_t, q3_t = q_bounds.iloc[0], q_bounds.iloc[1], q_bounds.iloc[2]
print(f"Q1≤{q1_t:.0f}mo  Q2≤{q2_t:.0f}mo  Q3≤{q3_t:.0f}mo")
print("Note: Duration quartile ≠ ExternalRiskEstimate (HELOC). [Comment 8]")

if len(set([q1_t, q2_t, q3_t])) < 3:
    med = dur.median()
    subgroups = {'Q1~Q2 (Short)': rejected_all[dur <= med],
                 'Q3~Q4 (Long)':  rejected_all[dur >  med]}
    print(f"  ※ Quartile collapse → binary split at {med:.0f}mo")
else:
    subgroups = {
        f'Q1 (≤{q1_t:.0f}mo)':           rejected_all[dur <= q1_t],
        f'Q2 ({q1_t:.0f}–{q2_t:.0f}mo)': rejected_all[(dur>q1_t)&(dur<=q2_t)],
        f'Q3 ({q2_t:.0f}–{q3_t:.0f}mo)': rejected_all[(dur>q2_t)&(dur<=q3_t)],
        f'Q4 (>{q3_t:.0f}mo)':            rejected_all[dur > q3_t],
    }
subgroups = {k: v for k, v in subgroups.items() if len(v) > 0}
sub_rows  = []
for grp, grp_df in subgroups.items():
    n_g = len(grp_df)
    ss, sp, iv, cv, br = 0, 0, 0, 0, 0; fc = []
    for i in tqdm(range(n_g), desc=grp):
        query = grp_df.iloc[i:i+1]
        pr    = build_permitted_range(query, actionable_features, THRESHOLD)
        try:
            res   = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED)
            cf_df = res.cf_examples_list[0].final_cfs_df
            r     = analyze_cf_paths(query, cf_df, target,
                                     immutable_features, actionable_features,
                                     scenario_c=True)
            sp += r['success_paths']; iv += r['imm_violation_paths']
            cv += r['cau_violation_paths']
            fc.extend(r['feat_changes_list'])
            if r['success_paths'] > 0: ss += 1
            if r['has_reliable_path']:  br += 1
        except Exception: continue
    vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
    sub_rows.append({
        'Subgroup': grp, 'N': n_g,
        'Avg_Duration(mo)': round(grp_df['Duration'].mean(), 1),
        'Sample_Success': ss,
        'RR(%)': round(ss/n_g*100 if n_g>0 else 0, 2),
        'Pathway_Reliable_RR(%)': round((1-vr)*(1-cvr)*100, 2),
        'Borrower_Reliable_RR(%)': round(br/n_g*100 if n_g>0 else 0, 2),
        'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
    })
df_sub = pd.DataFrame(sub_rows)
print(df_sub.to_string(index=False))
df_sub.to_csv('german_subgroup_analysis.csv', index=False)
if len(df_sub) >= 2:
    best  = df_sub.loc[df_sub['Pathway_Reliable_RR(%)'].idxmax()]
    worst = df_sub.loc[df_sub['Pathway_Reliable_RR(%)'].idxmin()]
    ratio = (best['Pathway_Reliable_RR(%)'] / worst['Pathway_Reliable_RR(%)']
             if worst['Pathway_Reliable_RR(%)'] > 0 else float('inf'))
    print(f"\n  Disparity: {ratio:.1f}-fold ({best['Subgroup']} vs {worst['Subgroup']})")
    print("  Descriptive only — gradient direction may differ from HELOC.")


# ── B-4. Algorithm Robustness ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-4: Algorithm Robustness (Random vs KD-Tree)")
print("=" * 70)
algo_rows = []
for method in ["random", "kdtree"]:
    exp_obj = dice_ml.Dice(d_dice, m_dice, method=method)
    for scenario in ["Scenario_A", "Scenario_C"]:
        is_c = (scenario == "Scenario_C")
        n = len(rejected_all)
        ss, sp, iv, cv, av, br, errors = 0, 0, 0, 0, 0, 0, 0; fc = []
        for i in tqdm(range(n), desc=f"{scenario}×{method}"):
            query = rejected_all.iloc[i:i+1]
            ftv   = "all" if scenario == "Scenario_A" else actionable_features
            pr    = None  if scenario == "Scenario_A" else \
                    build_permitted_range(query, actionable_features, THRESHOLD)
            try:
                kw = dict(total_CFs=TOTAL_CFS, desired_class="opposite",
                          features_to_vary=ftv, permitted_range=pr,
                          proximity_weight=0.5, sparsity_weight=1.0)
                if method == "random": kw['random_seed'] = RANDOM_SEED
                res   = exp_obj.generate_counterfactuals(query, **kw)
                cf_df = res.cf_examples_list[0].final_cfs_df
                act_f = actionable_features if is_c else None
                r     = analyze_cf_paths(query, cf_df, target,
                                         immutable_features, act_f,
                                         scenario_c=is_c)
                sp += r['success_paths']; iv += r['imm_violation_paths']
                cv += r['cau_violation_paths']; av += r['action_violation_paths']
                fc.extend(r['feat_changes_list'])
                if r['success_paths'] > 0: ss += 1
                if r['has_reliable_path']:  br += 1
            except Exception: errors += 1
        vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
        avr = av/sp if sp>0 else 0
        algo_rows.append({
            'Scenario': scenario, 'Method': method, 'N': n,
            'RR(%)': round(ss/n*100 if n>0 else 0, 2),
            'VR(%)': round(vr*100, 2), 'Causal_VR(%)': round(cvr*100, 2),
            'Action_VR(%)': round(avr*100, 2),
            'Pathway_Reliable_RR(%)': round((1-vr)*(1-cvr)*(1-avr)*100, 2),
            'Borrower_Reliable_RR(%)': round(br/n*100 if n>0 else 0, 2),
            'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
            'Errors': errors,
        })
df_algo = pd.DataFrame(algo_rows)
print(df_algo[['Scenario','Method','RR(%)','VR(%)','Action_VR(%)',
               'Pathway_Reliable_RR(%)','Borrower_Reliable_RR(%)','Errors']
              ].to_string(index=False))
df_algo.to_csv('german_method_comparison.csv', index=False)


# ── B-5. Representative Cases ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-5: Representative Case Extraction")
print("=" * 70)
cases = []
for i in range(min(100, len(rejected_all))):
    query = rejected_all.iloc[i:i+1]
    try:
        v_cf = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary="all", proximity_weight=0.5,
            random_seed=RANDOM_SEED).cf_examples_list[0].final_cfs_df
        pr   = build_permitted_range(query, actionable_features, THRESHOLD)
        p_cf = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary=actionable_features, permitted_range=pr,
            proximity_weight=0.5, sparsity_weight=1.0,
            random_seed=RANDOM_SEED).cf_examples_list[0].final_cfs_df
        if v_cf is None or v_cf.empty or p_cf is None or p_cf.empty: continue
        orig = query.iloc[0]
        for vi in range(len(v_cf)):
            v_row = v_cf.iloc[vi]
            if target in v_cf.columns and int(v_row[target]) == 1:
                if any(abs(v_row[f]-orig[f])>1e-5
                       for f in immutable_features if f in v_cf.columns):
                    for pi in range(len(p_cf)):
                        p_row = p_cf.iloc[pi]
                        if target in p_cf.columns and int(p_row[target]) == 1:
                            cases.append({'index': rejected_all.index[i],
                                          'original': orig.to_dict(),
                                          'vanilla_cf': v_row.to_dict(),
                                          'proposed_cf': p_row.to_dict()})
                            break
                    break
    except Exception: continue
    if len(cases) >= 3: break

if cases:
    key_vars = immutable_features[:3] + actionable_features[:3]
    for idx, case in enumerate(cases):
        print(f"\n[Case {idx+1}]  Index: {case['index']}")
        print(f"{'Variable':<25}{'Current':>10}{'Vanilla CF':>12}{'Proposed CF':>12}")
        print("-" * 63)
        for v in key_vars:
            o=case['original'].get(v,0); van=case['vanilla_cf'].get(v,0)
            prop=case['proposed_cf'].get(v,0)
            flag=" ⚠" if (v in immutable_features and abs(van-o)>1e-5) else ""
            print(f"{v:<25}{o:>10.2f}{van:>12.2f}{prop:>12.2f}{flag}")
        print(f"{'[Approval]':<25}{'Reject':>10}{'Approve':>12}{'Approve':>12}")


# ── B-6. Bootstrap ────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-6: Bootstrap Robustness (5 seeds × 3 scenarios)")
print("=" * 70)
bootstrap_rows = []; cal_rows = []; t_boot = time.time()

for s_idx, seed in enumerate(BOOTSTRAP_SEEDS):
    print(f"\n{'━'*60}")
    print(f"  Seed {seed}  ({s_idx+1}/{len(BOOTSTRAP_SEEDS)})")
    print(f"{'━'*60}")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=seed, stratify=y)
    bp = best_params_primary if seed == RANDOM_SEED else \
         run_optuna(X_tr, y_tr, N_OPTUNA_FAST, seed)[0]
    mdl_b, met_b = train_and_evaluate(
        X_tr, y_tr, X_te, y_te, bp, seed,
        plot_cal=(seed == RANDOM_SEED), label=f'seed{seed}')
    cal_rows.append({'Seed': seed, 'AUC-ROC': met_b['AUC-ROC'],
                     'Brier': met_b['Brier'], 'ECE': met_b['ECE']})
    rejected_b = X_te[mdl_b.predict(X_te) == 0].copy()
    print(f"  Rejected: {len(rejected_b)}")
    d_b   = dice_ml.Data(dataframe=dataset,
                         continuous_features=continuous_features,
                         outcome_name=target)
    exp_b = dice_ml.Dice(d_b,
                         dice_ml.Model(model=mdl_b, backend="sklearn"),
                         method="random")
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        row = run_scenario(sc, rejected_b, exp_b, THRESHOLD, seed,
                           desc_suffix=f'(seed={seed})')
        bootstrap_rows.append(row)

print(f"\n[B-6 elapsed] {(time.time()-t_boot)/60:.1f} min")
df_boot = pd.DataFrame(bootstrap_rows)
df_cal  = pd.DataFrame(cal_rows)
df_boot.to_csv('german_bootstrap_raw.csv', index=False)
df_cal.to_csv('german_bootstrap_calibration.csv', index=False)


# ── B-7. Summary + Transferability ────────────────────────────────────────────
print("\n" + "=" * 70)
print("B-7: Bootstrap Summary + Transferability")
print("=" * 70)

metrics_cols = ['RR(%)', 'VR(%)', 'Causal_VR(%)', 'Action_VR(%)',
                'Pathway_Reliable_RR(%)', 'Borrower_Reliable_RR(%)',
                'Avg_Features_Changed']

def ci95(vals):
    arr = np.array(vals, dtype=float)
    if len(arr) < 2: return arr[0], arr[0], arr[0]
    m = arr.mean()
    h = sp_stats.sem(arr) * sp_stats.t.ppf(0.975, df=len(arr)-1)
    return m, m-h, m+h

summary_rows = []
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    df_sc = df_boot[df_boot['Scenario'] == sc]
    row   = {'Scenario': sc}
    for col in metrics_cols:
        if col not in df_sc.columns: continue
        mean, lo, hi = ci95(df_sc[col].values)
        row[f'{col}_mean'] = round(mean, 2)
        row[f'{col}_CI95'] = f"[{lo:.2f}, {hi:.2f}]"
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    sr = df_summary[df_summary['Scenario'] == sc].iloc[0]
    print(f"\n── {sc} ──")
    for col in metrics_cols:
        if f'{col}_mean' not in sr: continue
        print(f"  {col:<28}: {sr[f'{col}_mean']:>7.2f}%  "
              f"95%CI {sr[f'{col}_CI95']}")
df_summary.to_csv('german_bootstrap_summary.csv', index=False)

print("\n── Calibration Metrics ──")
print(df_cal.to_string(index=False))
print(f"  Brier: {df_cal['Brier'].mean():.4f} ± {df_cal['Brier'].std():.4f}")
print(f"  ECE  : {df_cal['ECE'].mean():.4f} ± {df_cal['ECE'].std():.4f}")

# Transferability
print("\n── Transferability Comparison ──")
heloc_ref = pd.DataFrame([
    {'Dataset':'HELOC','Scenario':'Scenario_A',
     'RR(%)':100.00,'VR(%)':94.11,'Causal_VR(%)':11.49,'Pathway_Reliable_RR(%)':5.22},
    {'Dataset':'HELOC','Scenario':'Scenario_B',
     'RR(%)':36.87,'VR(%)':0.00,'Causal_VR(%)':28.22,'Pathway_Reliable_RR(%)':26.47},
    {'Dataset':'HELOC','Scenario':'Scenario_C',
     'RR(%)':11.05,'VR(%)':0.00,'Causal_VR(%)':46.92,'Pathway_Reliable_RR(%)':5.87},
])
german_ref = df_scenarios[['Scenario','RR(%)','VR(%)','Causal_VR(%)',
                            'Pathway_Reliable_RR(%)']].copy()
german_ref.insert(0, 'Dataset', 'German Credit')
transfer = pd.concat([heloc_ref, german_ref], ignore_index=True)
print(transfer.to_string(index=False))
transfer.to_csv('german_transferability_comparison.csv', index=False)

# Bootstrap plot
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = {'Scenario_A':'#C00000','Scenario_B':'#ED7D31','Scenario_C':'#5B9BD5'}
for ax, col, ylabel in zip(
    axes,
    ['Pathway_Reliable_RR(%)', 'Borrower_Reliable_RR(%)', 'RR(%)'],
    ['Pathway Reliable RR (%)', 'Borrower Reliable RR (%)', 'RR (%)']
):
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        vals = df_boot[df_boot['Scenario'] == sc][col].values
        mean, lo, hi = ci95(vals)
        ax.errorbar(x=sc.replace('Scenario_','Sc.'), y=mean,
                    yerr=[[mean-lo],[hi-mean]], fmt='o', capsize=6,
                    color=colors[sc], markersize=8)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(col.replace('(%)',''), fontsize=10, fontweight='bold')
    ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.suptitle(f'Bootstrap Robustness — German Credit ({len(BOOTSTRAP_SEEDS)} seeds)\n'
             'Error bars: 95% CI', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('bootstrap_robustness_german.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: bootstrap_robustness_german.png")

# ── Final ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(f"German Credit All Done — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)
for f in ['german_model_performance.csv','german_scenario_comparison.csv',
          'german_sensitivity_analysis.csv','german_subgroup_analysis.csv',
          'german_method_comparison.csv','german_bootstrap_raw.csv',
          'german_bootstrap_summary.csv','german_bootstrap_calibration.csv',
          'german_transferability_comparison.csv',
          'german_calibration_seed42.png','bootstrap_robustness_german.png']:
    print(f"  {f}")

██████████████████████████████████████████████████████████████████████
  PART B — 전체 실험 시작
██████████████████████████████████████████████████████████████████████

B-1: Primary Scenario Simulation (seed 42, full test set)


Scenario_A: 100%|██████████████████████████████████████████████████████████████████████| 59/59 [00:06<00:00,  8.62it/s]


  [Scenario_A] RR=100.0%  VR=21.6%  Causal_VR=0.0%  Action_VR=35.2%  Path_RRR=50.8%  Bor_RRR=96.6%


Scenario_B: 100%|██████████████████████████████████████████████████████████████████████| 59/59 [00:07<00:00,  7.63it/s]


  [Scenario_B] RR=100.0%  VR=0.0%  Causal_VR=0.8%  Action_VR=43.6%  Path_RRR=55.9%  Bor_RRR=98.3%


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C(seed=2024):  98%|█████████████████████████████████████████████████████████▋ | 42/43 [00:08<00:00,  5.01it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



Scenario_C(seed=2024): 100%|███████████████████████████████████████████████████████████| 43/43 [00:08<00:00,  5.08it/s]


  [Scenario_C] RR=86.0%  VR=0.0%  Causal_VR=21.8%  Action_VR=0.0%  Path_RRR=78.2%  Bor_RRR=83.7%

[B-6 elapsed] 3.7 min

B-7: Bootstrap Summary + Transferability

── Scenario_A ──
  RR(%)                       :  100.00%  95%CI [100.00, 100.00]
  VR(%)                       :   24.89%  95%CI [22.18, 27.60]
  Causal_VR(%)                :    0.31%  95%CI [-0.27, 0.88]
  Action_VR(%)                :   33.54%  95%CI [30.26, 36.82]
  Pathway_Reliable_RR(%)      :   49.74%  95%CI [47.62, 51.85]
  Borrower_Reliable_RR(%)     :   93.26%  95%CI [85.52, 101.00]
  Avg_Features_Changed        :    1.84%  95%CI [1.78, 1.89]

── Scenario_B ──
  RR(%)                       :  100.00%  95%CI [100.00, 100.00]
  VR(%)                       :    0.00%  95%CI [0.00, 0.00]
  Causal_VR(%)                :    0.72%  95%CI [0.16, 1.27]
  Action_VR(%)                :   41.11%  95%CI [37.74, 44.48]
  Pathway_Reliable_RR(%)      :   58.46%  95%CI [55.07, 61.86]
  Borrower_Reliable_RR(%)     :   96.22%  95%CI 